# Notebook 03 — Kiểm thử và tái lập kết quả K-Medoids

Các kiểm tra dùng API hiện tại. Cấu hình hiện hành có 12 đặc trưng, 21 cột Manhattan sau tiền xử lý, cùng hai cột tham chiếu cp và target.

Miền mã theo nguồn đang được xác nhận riêng. Mục dữ liệu bên dưới ghi nhận mã ngoài miền hiện tại và kết quả lọc theo nguồn dưới dạng cảnh báo để theo dõi. Các kiểm tra tái lập cấu hình dùng toàn bộ nguồn.

Cách chạy: chạy Notebook 01 để tạo dữ liệu và metric, chạy Notebook 02 để lưu ví dụ thư viện 30 hồ sơ, rồi chạy hết Notebook 03 bằng kernel của dự án. Những phép thử ghi/đọc dùng thư mục tạm. Notebook kiểm tra mã băm đầu vào ở cuối.

In [1]:
import sys
from pathlib import Path
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from IPython.display import display
from tempfile import TemporaryDirectory

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src/kmedoids.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.kmedoids import KMedoids
from src.data import ALL_COLS, FEATURE_COLS, REFERENCE_COLS, TARGET_COL, DataQualityAnalyzer, HeartDataLoader
from src.preprocessing import DistanceCalculator, FeaturePreprocessor, ProcessedDataSaver
from src.library_model import fit_kmedoids
from src.evaluation import ClusterEvaluator, MetricsRunner

RAW_PATH = PROJECT_ROOT / "data/raw/heart_disease_whole.csv"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
METRICS_DIR = PROJECT_ROOT / "outputs/metrics"
EXAMPLE_DIR = PROJECT_ROOT / "outputs/library_30_records"
TEST_RESULTS = []
PROTECTED_PATHS = [
    RAW_PATH,
    *(PROCESSED_DIR / name for name in ["heart_dedup.csv", "X_manhattan.csv", "target.csv",
                                       "reference.csv", "gower_distance.npy"]),
    *(METRICS_DIR / name for name in ["k_selection.csv", "labels.csv", "medoids.csv",
                                     "cross_method_ari.csv", "run_config.json"]),
    *(EXAMPLE_DIR / name for name in ["assignments.csv", "clusters.csv", "medoids.csv", "summary.json"]),
    PROJECT_ROOT / "notebooks/02_experiment_analysis.ipynb",
]
missing = [str(p.relative_to(PROJECT_ROOT)) for p in PROTECTED_PATHS if not p.is_file()]
if missing:
    raise FileNotFoundError("Cần chạy Notebook 01 và 02. Thiếu: " + ", ".join(missing))
FILE_HASHES_BEFORE = {p: sha256(p.read_bytes()).hexdigest() for p in PROTECTED_PATHS}

print("Project root:", PROJECT_ROOT)
print("Các đầu vào và kết quả thực nghiệm hiện có được mở ở chế độ đọc.")

Project root: C:\Users\havuh\OneDrive\Desktop\HocTap\Clustering_ of patients_with heart_disease_KMedoids
Các đầu vào và kết quả thực nghiệm hiện có được mở ở chế độ đọc.


## 1. Chạy từng ca kiểm thử

PASS là đạt. FAIL làm notebook báo lỗi ở phần tổng kết. WARN ghi lại các vấn đề dữ liệu hoặc chức năng ngoài cấu hình toàn bộ nguồn, để theo dõi riêng.

In [2]:
def test_case(name, fn, required=True, note=""):
    try:
        detail = fn()
        TEST_RESULTS.append({"test":name,"status":"PASS","required":required,"detail":str(detail),"note":note})
        print(f"PASS | {name} | {detail}")
        return True
    except Exception as exc:
        status = "FAIL" if required else "WARN"
        TEST_RESULTS.append({"test":name,"status":status,"required":required,"detail":f"{type(exc).__name__}: {exc}","note":note})
        print(f"{status} | {name} | {type(exc).__name__}: {exc}")
        return False

def expect_raises(exception_type, fn):
    try:
        fn()
    except exception_type as exc:
        return f"{type(exc).__name__}: {exc}"
    except Exception as exc:
        expected = getattr(exception_type,"__name__",str(exception_type))
        raise AssertionError(f"Expected {expected}; got {type(exc).__name__}: {exc}") from exc
    expected = getattr(exception_type,"__name__",str(exception_type))
    raise AssertionError(f"Expected {expected}; no exception was raised")

## 2. Unit tests cho custom K-Medoids
Bao gồm ví dụ 30 điểm tính tay, medoid thuộc dữ liệu, seed, `fit_predict`/`predict`, input không hợp lệ, max_iter, record trùng và các trường hợp hòa.

In [3]:
# Các unit test chạy bằng assert, không cần pytest.
X30 = [[v] for v in list(range(1, 16)) + list(range(100, 115))]

def test_30_record_hand_example():
    m = KMedoids(2, metric="manhattan", initial_medoids=[0, 15]).fit(X30)
    assert len(X30) == 30
    assert m.medoid_indices_ == [7, 22]
    assert m.medoids_ == [[8.0], [107.0]]
    assert m.labels_ == [0] * 15 + [1] * 15
    assert m.cluster_sizes_ == [15, 15]
    assert np.isclose(m.inertia_, 112.0)
    assert m.n_iter_ == 2 and m.converged_
    return f"medoids={m.medoid_indices_}; objective={m.inertia_}; iterations={m.n_iter_}"

def test_medoids_are_actual_input_records():
    X = [[0, 0], [0, 1], [9, 9], [10, 9]]
    m = KMedoids(2, metric="manhattan", random_state=42).fit(X)
    observed = {tuple(map(float, row)) for row in X}
    assert all(tuple(row) in observed for row in m.medoids_)
    assert all(0 <= idx < len(X) for idx in m.medoid_indices_)
    return f"medoid rows={m.medoid_indices_} are members of X"

def test_seed_reproducibility():
    X = [[i, (i * 7) % 11] for i in range(20)]
    a = KMedoids(3, metric="manhattan", random_state=42).fit(X)
    b = KMedoids(3, metric="manhattan", random_state=42).fit(X)
    assert a.medoid_indices_ == b.medoid_indices_
    assert a.labels_ == b.labels_
    assert np.isclose(a.inertia_, b.inertia_)
    return f"same seed -> medoids={a.medoid_indices_}, labels/objective reproducible"

def test_euclidean_and_manhattan_metrics():
    X=[[0],[1],[9],[10]]
    expected=[0,0,1,1]
    for metric in ["euclidean","manhattan"]:
        m=KMedoids(2,metric=metric,initial_medoids=[0,2]).fit(X)
        assert m.labels_==expected and m.medoid_indices_==[0,2]
        assert np.isclose(m.inertia_,2.0) and m.converged_
    return "Euclidean and Manhattan produce expected labels, medoids and objective"

def test_fit_predict_predict_and_new_rows():
    X = [[0], [1], [2], [10], [11], [12]]
    fit = KMedoids(2, metric="manhattan", random_state=42).fit(X)
    fit_predict = KMedoids(2, metric="manhattan", random_state=42).fit_predict(X)
    assert fit_predict == fit.labels_
    assert fit.predict(X) == fit.labels_
    assert fit.predict([[0]]) == [fit.labels_[0]]
    assert fit.predict([[12]]) == [fit.labels_[-1]]
    return f"fit_predict matches labels_; predict(X_train/new) passed"

def test_predict_errors():
    expect_raises(RuntimeError, lambda: KMedoids(2).predict([[1, 2]]))
    fitted = KMedoids(2, random_state=0).fit([[0, 0], [1, 1], [9, 9]])
    expect_raises(ValueError, lambda: fitted.predict([[1]]))
    return "predict before fit and wrong feature count rejected"

def test_constructor_parameter_validation():
    expect_raises(TypeError, lambda: KMedoids(True))
    expect_raises(TypeError, lambda: KMedoids(2.0))
    expect_raises(ValueError, lambda: KMedoids(0))
    expect_raises(ValueError, lambda: KMedoids(2, max_iter=0))
    expect_raises(ValueError, lambda: KMedoids(2, metric="cosine"))
    return "invalid k type/value, max_iter, and metric rejected"

def test_input_validation_and_invalid_k():
    for bad in [[], [[], []], [[1], [2, 3]], [[np.nan], [1]], [[np.inf], [1]], [["x"], [1]]]:
        expect_raises((TypeError, ValueError), lambda bad=bad: KMedoids(1).fit(bad))
    expect_raises(ValueError, lambda: KMedoids(3).fit([[1], [2]]))
    expect_raises(ValueError, lambda: KMedoids(2, initial_medoids=[0, 0]).fit([[1], [2]]))
    expect_raises(ValueError, lambda: KMedoids(2, initial_medoids=[0, 3]).fit([[1], [2]]))
    expect_raises(ValueError, lambda: KMedoids(2, initial_medoids=[0]).fit([[1], [2]]))
    expect_raises(TypeError, lambda: KMedoids(2, initial_medoids=[0, 1.0]).fit([[1], [2]]))
    return "empty/ragged/non-numeric/non-finite data, bad k and medoid indices rejected"

def test_max_iter_limit():
    m = KMedoids(2, metric="manhattan", max_iter=1, initial_medoids=[0, 15]).fit(X30)
    assert m.n_iter_ == 1 and not m.converged_
    assert m.medoid_indices_ == [7, 22]
    assert m.labels_ == [0] * 15 + [1] * 15
    return f"max_iter=1 -> n_iter={m.n_iter_}, converged={m.converged_}; final labels/objective recomputed"

def test_duplicate_rows_and_assignment_tie():
    dup = KMedoids(2, metric="manhattan", initial_medoids=[0, 2]).fit([[0], [0], [10], [10]])
    assert dup.labels_ == [0, 0, 1, 1] and dup.medoid_indices_ == [0, 2] and dup.inertia_ == 0
    tie = KMedoids(2, metric="manhattan", initial_medoids=[0, 1]).fit([[0], [2], [1]])
    assert tie.labels_ == [0, 1, 0]  # equidistant sample -> lower cluster id
    assert tie.medoid_indices_ == [0, 1]
    return "duplicate points preserved; assignment tie resolves to lower cluster id"

def test_all_missing_imputation_fallbacks():
    frame=pd.DataFrame({"cp":[np.nan,np.nan],"age":[np.nan,np.nan]})
    loader = HeartDataLoader(RAW_PATH)
    filled = loader._impute_mode(frame, cols=["cp"])
    filled = loader._impute_median(filled, cols=["age"])
    assert filled["cp"].tolist()==[0,0] and loader.fill_categorical["cp"]==0
    assert filled["age"].tolist()==[0.0,0.0] and loader.fill_numeric["age"]==0.0
    return "all-NaN categorical/numeric columns use documented zero fallback"

def test_medoid_update_tie_policy():
    m = KMedoids(2, metric="manhattan", initial_medoids=[0, 2]).fit([[0], [2], [10]])
    assert m.medoid_indices_ == [0, 2]  # tied cost keeps old medoid when it is among minima
    assert m.converged_
    return "equal medoid costs keep previous medoid when tied"

CUSTOM_TESTS = [
    ("30-record hand calculation", test_30_record_hand_example),
    ("medoid must be observed record", test_medoids_are_actual_input_records),
    ("random seed reproducibility", test_seed_reproducibility),
    ("Euclidean and Manhattan distances", test_euclidean_and_manhattan_metrics),
    ("fit_predict and predict", test_fit_predict_predict_and_new_rows),
    ("predict input errors", test_predict_errors),
    ("constructor validation", test_constructor_parameter_validation),
    ("fit data/k/initial medoid validation", test_input_validation_and_invalid_k),
    ("max_iter and final recomputation", test_max_iter_limit),
    ("duplicate rows and assignment tie", test_duplicate_rows_and_assignment_tie),
    ("all-missing imputation fallback", test_all_missing_imputation_fallbacks),
    ("medoid-update tie policy", test_medoid_update_tie_policy),
]

for test_name, test_fn in CUSTOM_TESTS:
    test_case("CUSTOM / " + test_name, test_fn)


PASS | CUSTOM / 30-record hand calculation | medoids=[7, 22]; objective=112.0; iterations=2
PASS | CUSTOM / medoid must be observed record | medoid rows=[0, 3] are members of X
PASS | CUSTOM / random seed reproducibility | same seed -> medoids=[6, 2, 13], labels/objective reproducible
PASS | CUSTOM / Euclidean and Manhattan distances | Euclidean and Manhattan produce expected labels, medoids and objective
PASS | CUSTOM / fit_predict and predict | fit_predict matches labels_; predict(X_train/new) passed
PASS | CUSTOM / predict input errors | predict before fit and wrong feature count rejected
PASS | CUSTOM / constructor validation | invalid k type/value, max_iter, and metric rejected
PASS | CUSTOM / fit data/k/initial medoid validation | empty/ragged/non-numeric/non-finite data, bad k and medoid indices rejected
PASS | CUSTOM / max_iter and final recomputation | max_iter=1 -> n_iter=1, converged=False; final labels/objective recomputed
PASS | CUSTOM / duplicate rows and assignment tie |

## 3. Tải dữ liệu, làm sạch và tách cột tham chiếu

Kiểm tra cấu hình hiện tại dùng toàn bộ nguồn. Ghi nhận riêng các mã ngoài miền sau quy đổi và khả năng lọc một nguồn bằng API.

In [4]:
def test_loader_reads_all_sources():
    raw = pd.read_csv(RAW_PATH)
    loader = HeartDataLoader(RAW_PATH)
    df = loader.clean()
    assert len(raw) == len(loader.raw) == len(df) == 2283
    assert df.columns.tolist() == ALL_COLS and df.notna().all().all()
    return f"{raw.source.nunique()} sources; {len(df)} cleaned rows"


def test_loader_rejects_missing_file():
    return expect_raises(FileNotFoundError, lambda: HeartDataLoader(
        PROJECT_ROOT / "__missing_test_file.csv").clean())


def test_harmonization_is_copy_and_source_aware():
    fixture = pd.DataFrame({
        "cp": [1, 1], "slope": [1, 1], "thal": [0, 0],
        "source": ["kaggle_heart_1025", "uci_multi_cleveland"],
    })
    original = fixture.copy()
    result = HeartDataLoader._harmonize_to_uci_encoding(fixture)
    pd.testing.assert_frame_equal(fixture, original)
    assert result[["cp", "slope", "thal"]].values.tolist() == [[2, 2, 3], [1, 1, 0]]
    return "Current source-specific transform; input frame unchanged"


def test_duplicate_abnormal_imputation_and_split():
    fixture = pd.DataFrame({
        "age":[40,40,50], "trestbps":[120,120,np.nan], "chol":[200,200,220],
        "thalach":[150,150,160], "oldpeak":[0,0,1], "cp":[1,1,9],
        "restecg":[0,0,1], "slope":[1,1,2], "ca":[0,0,1], "thal":[3,3,6],
        "sex":[0,0,1], "fbs":[0,0,1], "exang":[0,0,1], "target":[0,0,1],
    })
    with TemporaryDirectory(prefix="kmedoids-loader-") as temp:
        path = Path(temp) / "fixture.csv"
        fixture.to_csv(path, index=False)
        loader = HeartDataLoader(path)
        df = loader.clean()
        assert len(df) == 2
        assert df["cp"].tolist() == [1,1] and loader.fill_categorical["cp"] == 1
        assert df["trestbps"].tolist() == [120,120] and loader.fill_numeric["trestbps"] == 120
        X, y = loader.split_features_target(df)
        ref = loader.split_reference(df)
        assert X.columns.tolist() == FEATURE_COLS and X.shape == (2,12)
        assert ref.columns.tolist() == [*REFERENCE_COLS,TARGET_COL]
        pd.testing.assert_series_equal(y, ref[TARGET_COL])
        assert DataQualityAnalyzer.report(df)["n_missing"] == 0
    return "Dedup, mode/median fill, 12 features and both reference columns"


def check_codes_outside_current_domain():
    outliers = DataQualityAnalyzer.report(HeartDataLoader(RAW_PATH).raw)["unexpected_values"]
    if outliers:
        raise ValueError(f"Mã ngoài miền sau quy đổi: {outliers}")
    return "Các mã sau quy đổi nằm trong miền hiện tại."


def check_optional_source_filter():
    raw = pd.read_csv(RAW_PATH)
    source = raw["source"].iloc[0]
    with TemporaryDirectory(prefix="kmedoids-source-filter-") as temp:
        path = Path(temp) / "subset.csv"
        raw.loc[raw["source"].eq(source)].to_csv(path, index=False)
        expected = HeartDataLoader(path).clean()
        filtered = HeartDataLoader(RAW_PATH).clean(sources=[source])
        pd.testing.assert_frame_equal(filtered, expected)
    return f"Source {source}: {len(filtered)} rows"


for name, fn in [
    ("loader all-source", test_loader_reads_all_sources),
    ("missing raw file", test_loader_rejects_missing_file),
    ("source-specific harmonization", test_harmonization_is_copy_and_source_aware),
    ("dedup/imputation/split", test_duplicate_abnormal_imputation_and_split),
]:
    test_case("DATA / " + name, fn)
test_case("DATA / codes outside current domain", check_codes_outside_current_domain, required=False,
          note="Xác nhận ý nghĩa mã theo nguồn trước khi chốt diễn giải cp.")
test_case("DATA / optional source filter", check_optional_source_filter, required=False,
          note="Cấu hình thực nghiệm dùng toàn bộ nguồn.")

PASS | DATA / loader all-source | 7 sources; 2283 cleaned rows
PASS | DATA / missing raw file | FileNotFoundError: Không tìm thấy dữ liệu: C:\Users\havuh\OneDrive\Desktop\HocTap\Clustering_ of patients_with heart_disease_KMedoids\__missing_test_file.csv
PASS | DATA / source-specific harmonization | Current source-specific transform; input frame unchanged
PASS | DATA / dedup/imputation/split | Dedup, mode/median fill, 12 features and both reference columns
WARN | DATA / codes outside current domain | ValueError: Mã ngoài miền sau quy đổi: {'cp': {5: 424}, 'slope': {4.0: 183}, 'ca': {4.0: 4}}
WARN | DATA / optional source filter | ValueError: File không có cột 'source' để filter


False

## 4. Tiền xử lý, metric và adapter thư viện

Dùng dữ liệu tổng hợp nhỏ để kiểm tra biểu diễn Manhattan, Gower, chỉ số đánh giá và lưu/đọc đầu ra trong thư mục tạm.

In [5]:
def test_manhattan_preprocessing_schema():
    frame = pd.DataFrame({
        "age":[40,50,60,70],"trestbps":[120,130,140,150],"chol":[200,210,220,230],"thalach":[160,150,140,130],"oldpeak":[0,.5,1,1.5],
        "cp":[1,2,3,4],"restecg":[0,1,2,0],"slope":[1,2,3,1],"ca":[0,1,2,3],"thal":[3,6,7,3],
        "sex":[0,1,0,1],"fbs":[0,0,1,1],"exang":[0,1,0,1]})
    matrix, names, transformer = FeaturePreprocessor().fit_transform(frame)
    assert matrix.shape == (4,21) and len(names)==21 and np.isfinite(matrix).all()
    assert not any(c.startswith("cp_") or c == TARGET_COL for c in names)
    other, _, _ = FeaturePreprocessor().fit_transform(frame.assign(cp=[4,4,4,4]))
    np.testing.assert_allclose(other, matrix)
    return f"fixture produces {matrix.shape[1]} finite Manhattan features"

def test_preprocessing_missing_columns_and_gower_properties():
    expect_raises(ValueError,lambda:FeaturePreprocessor().fit_transform(pd.DataFrame({"age":[1]})))
    frame=pd.DataFrame({
        "age":[40,50,60],"trestbps":[120,130,140],"chol":[200,210,220],"thalach":[160,150,140],"oldpeak":[0,.5,1],
        "cp":[1,2,4],"restecg":[0,1,2],"slope":[1,2,3],"ca":[0,1,3],"thal":[3,6,7],"sex":[0,1,0],"fbs":[0,1,0],"exang":[0,1,0]})
    D=DistanceCalculator.gower(frame)
    np.testing.assert_allclose(DistanceCalculator.gower(frame.assign(cp=[4,4,4])), D)
    assert D.shape==(3,3) and np.isfinite(D).all()
    assert np.allclose(D,D.T) and np.allclose(np.diag(D),0)
    assert (D>=0).all() and (D<=1).all()
    expect_raises(ValueError,lambda:DistanceCalculator.gower(frame.assign(age=[40,np.nan,60])))
    return f"Gower symmetric, diagonal=0, range=[{D.min():.3f},{D.max():.3f}]"

def test_metrics_known_fixtures():
    labels=np.array([0,0,1,1]); y=pd.Series([0,0,1,1],name=TARGET_COL)
    ext=ClusterEvaluator.posthoc_check(labels,y)
    assert np.isclose(ext["ARI"],1) and np.isclose(ext["purity"],1)
    assert ClusterEvaluator.cluster_sizes(labels).to_dict()=={0:2,1:2}
    assert ClusterEvaluator.compare_labelings(labels,labels)==1.0
    assert np.isnan(ClusterEvaluator.silhouette(np.zeros((4,1)),np.zeros(4),metric="euclidean"))
    D=np.array([[0,1,9,10],[1,0,8,9],[9,8,0,1],[10,9,1,0]],float)
    score=ClusterEvaluator.silhouette(D,labels,metric="precomputed")
    assert 0 < score <= 1
    return f"known ARI/purity=1; precomputed silhouette={score:.3f}"

def test_profile_and_target_crosstab_fixture():
    frame=pd.DataFrame({"age":[20,30,50,60],"trestbps":[100,110,140,150],"chol":[180,190,230,240],"thalach":[180,170,140,130],"oldpeak":[0,.1,2,3],"sex":[0,1,1,1],"fbs":[0,0,1,1],"exang":[0,0,1,1],"target":[0,0,1,1]})
    labels=np.array([0,0,1,1])
    profile=ClusterEvaluator.cluster_profile(frame,labels)
    assert profile["n"].tolist()==[2,2] and profile["target_rate"].tolist()==[0.0,1.0]
    ct=ClusterEvaluator.target_crosstab(labels,frame[TARGET_COL])
    assert np.allclose(ct["tỉ_lệ_bệnh"].to_numpy(),[0,1])
    return "cluster sizes and post-hoc rates match fixture"

def test_library_adapter_modes():
    X=np.array([[0.,0.],[0.,1.],[10.,10.],[10.,11.]])
    D=np.abs(X[:,None,:]-X[None,:,:]).sum(axis=2)
    a=fit_kmedoids(X,2,metric="manhattan",random_state=42)
    b=fit_kmedoids(D,2,metric="precomputed",random_state=42)
    assert a.labels.shape==(4,) and a.medoid_indices.shape==(2,) and np.isfinite(a.inertia)
    assert b.labels.shape==(4,) and b.medoid_indices.shape==(2,) and np.isfinite(b.inertia)
    expect_raises(ValueError,lambda:fit_kmedoids(np.zeros((3,2)),2,metric="precomputed"))
    return f"feature/precomputed adapter outputs have {len(a.labels)} labels"

def test_select_k_pipeline_fixture():
    X=np.array([[0.,0.],[0.,1.],[1.,0.],[9.,9.],[10.,9.],[9.,10.]])
    best,table,results=ClusterEvaluator.select_k(X,metric="manhattan",k_range=[2,3],random_state=42)
    assert best==2 and set(table.index)=={2,3} and set(results)=={2,3}
    assert {"silhouette","inertia","min_cluster_size","max_cluster_size"}.issubset(table.columns)
    assert all(np.isfinite(table[c]).all() for c in ["silhouette","inertia"])
    assert all(len(r.labels)==len(X) and len(r.medoid_indices)==k for k,r in results.items())
    assert table.loc[2,"silhouette"]>table.loc[3,"silhouette"]
    return f"select_k chose k={best}; scores={table.silhouette.round(3).to_dict()}"

def test_processed_artifact_roundtrip_in_temporary_directory():
    frame=pd.DataFrame({
        "age":[30,35,40,45,50,55],"trestbps":[110,115,120,125,130,135],"chol":[180,185,190,195,200,205],"thalach":[180,175,170,165,160,155],"oldpeak":[0,.1,.2,.3,.4,.5],
        "cp":[1,2,3,4,1,2],"restecg":[0,1,2,0,1,2],"slope":[1,2,3,1,2,3],"ca":[0,1,2,3,0,1],"thal":[3,6,7,3,6,7],
        "sex":[0,1,0,1,0,1],"fbs":[0,0,1,1,0,1],"exang":[0,1,0,1,0,1],"target":[0,1,0,1,0,1]})
    X_mat,names,_=FeaturePreprocessor().fit_transform(frame[FEATURE_COLS])
    D=DistanceCalculator.gower(frame[FEATURE_COLS])
    with TemporaryDirectory(prefix="heart_kmedoids_test_processed_") as temp:
        paths=ProcessedDataSaver.save(frame,X_mat,names,D,Path(temp)/"processed")
        base=pd.read_csv(paths["dedup"]); X_saved=pd.read_csv(paths["X"]); y=pd.read_csv(paths["target"])
        D_saved=np.load(paths["gower"])
        assert base.equals(frame) and X_saved.columns.tolist()==names
        assert np.allclose(X_saved.to_numpy(),X_mat) and y[TARGET_COL].equals(frame[TARGET_COL])
        assert np.allclose(D_saved,D)
        pd.testing.assert_frame_equal(pd.read_csv(paths["reference"]), frame[[*REFERENCE_COLS,TARGET_COL]])
    return "save_processed write/read roundtrip matches fixture; temporary files cleaned"

def test_metrics_pipeline_writes_and_reads_temp_outputs():
    # Nine distinct synthetic rows allow the project's fixed k=2..8 sweep.
    frame=pd.DataFrame({
        "age":[31,34,38,42,47,53,58,64,70],"trestbps":[108,116,124,132,140,148,156,164,172],"chol":[175,186,197,208,219,230,241,252,263],"thalach":[190,182,174,166,158,150,142,134,126],"oldpeak":[0,.1,.2,.3,.4,.5,.6,.7,.8],
        "cp":[1,2,3,4,1,2,3,4,1],"restecg":[0,1,2,0,1,2,0,1,2],"slope":[1,2,3,1,2,3,1,2,3],"ca":[0,1,2,3,0,1,2,3,0],"thal":[3,6,7,3,6,7,3,6,7],
        "sex":[0,1,0,1,0,1,0,1,0],"fbs":[0,0,1,1,0,0,1,1,0],"exang":[0,1,0,1,0,1,0,1,0],"target":[0,1,0,1,0,1,0,1,0]})
    with TemporaryDirectory(prefix="heart_kmedoids_test_metrics_") as temp:
        paths=MetricsRunner(temp).run(frame,extra_config={"test_only":True})
        expected={"k_selection","labels","medoids","cross_method_ari","run_config"}
        assert set(paths)==expected and all(Path(path).is_file() for path in paths.values())
        metrics=pd.read_csv(paths["k_selection"]); labels=pd.read_csv(paths["labels"])
        medoids=pd.read_csv(paths["medoids"]); cross=pd.read_csv(paths["cross_method_ari"])
        config=json.loads(Path(paths["run_config"]).read_text(encoding="utf-8"))
        assert len(metrics)==14 and set(metrics["k"])==set(range(2,9))
        assert labels.shape==(len(frame),14) and len(medoids)==2*sum(range(2,9))
        assert cross["k"].tolist()==list(range(2,9))
        assert config["n_rows"]==len(frame) and config["test_only"] is True
        assert config["held_out_columns"]==[*REFERENCE_COLS,TARGET_COL]
        assert {"ARI_cp","purity_cp","ARI_target","purity_target"}.issubset(metrics.columns)
    return "run_and_save_metrics outputs/files/row counts/config roundtrip passed; temporary outputs cleaned"

PREP_EVAL_TESTS=[
    ("Manhattan feature schema",test_manhattan_preprocessing_schema),
    ("Gower matrix validation",test_preprocessing_missing_columns_and_gower_properties),
    ("metrics known fixtures",test_metrics_known_fixtures),
    ("cluster profile/crosstab",test_profile_and_target_crosstab_fixture),
    ("library feature/precomputed adapter",test_library_adapter_modes),
    ("select_k pipeline",test_select_k_pipeline_fixture),
    ("processed artifact roundtrip (temporary directory)",test_processed_artifact_roundtrip_in_temporary_directory),
    ("metrics save/read integration (temporary directory)",test_metrics_pipeline_writes_and_reads_temp_outputs),
]
for test_name,test_fn in PREP_EVAL_TESTS:
    test_case("PREPROCESS/EVALUATION/LIBRARY / "+test_name,test_fn)

PASS | PREPROCESS/EVALUATION/LIBRARY / Manhattan feature schema | fixture produces 21 finite Manhattan features
PASS | PREPROCESS/EVALUATION/LIBRARY / Gower matrix validation | Gower symmetric, diagonal=0, range=[0.000,0.792]
PASS | PREPROCESS/EVALUATION/LIBRARY / metrics known fixtures | known ARI/purity=1; precomputed silhouette=0.889
PASS | PREPROCESS/EVALUATION/LIBRARY / cluster profile/crosstab | cluster sizes and post-hoc rates match fixture
PASS | PREPROCESS/EVALUATION/LIBRARY / library feature/precomputed adapter | feature/precomputed adapter outputs have 4 labels
PASS | PREPROCESS/EVALUATION/LIBRARY / select_k pipeline | select_k chose k=2; scores={2: 0.926, 3: 0.546}
PASS | PREPROCESS/EVALUATION/LIBRARY / processed artifact roundtrip (temporary directory) | save_processed write/read roundtrip matches fixture; temporary files cleaned
PASS | PREPROCESS/EVALUATION/LIBRARY / metrics save/read integration (temporary directory) | run_and_save_metrics outputs/files/row counts/config

## 5. Đối chiếu kết quả đã lưu và ví dụ 30 hồ sơ

Tạo lại dữ liệu đã xử lý trong bộ nhớ, kiểm tra lần chạy tự cài đặt với toàn bộ dữ liệu và so sánh 30 hồ sơ với kết quả Notebook 02.

In [6]:
def test_saved_artifacts_and_config_are_aligned():
    base = pd.read_csv(PROCESSED_DIR/"heart_dedup.csv")
    X = pd.read_csv(PROCESSED_DIR/"X_manhattan.csv")
    reference = pd.read_csv(PROCESSED_DIR/"reference.csv")
    target = pd.read_csv(PROCESSED_DIR/"target.csv")
    labels = pd.read_csv(METRICS_DIR/"labels.csv")
    medoids = pd.read_csv(METRICS_DIR/"medoids.csv")
    metrics = pd.read_csv(METRICS_DIR/"k_selection.csv")
    config = json.loads((METRICS_DIR/"run_config.json").read_text(encoding="utf-8"))
    assert config["n_rows"] == len(base) == len(reference) == len(labels) == 2283
    assert X.shape == (2283,21) and config["n_features_manhattan"] == 21
    assert config["n_features_raw"] == 12 and config["source_filter"] is None
    assert config["held_out_columns"] == [*REFERENCE_COLS,TARGET_COL]
    assert base.columns.tolist() == ALL_COLS
    pd.testing.assert_frame_equal(reference, base[[*REFERENCE_COLS,TARGET_COL]])
    pd.testing.assert_frame_equal(target, reference[[TARGET_COL]])
    assert {"ARI_cp","purity_cp","ARI_target","purity_target"}.issubset(metrics.columns)
    assert set(labels.columns) == {f"{d}_k{k}" for d in ["manhattan","gower"] for k in range(2,9)}
    assert medoids["medoid_index"].between(0,len(base)-1).all()
    return "2,283 aligned rows, 21 encoded columns, cp/target separate"


def test_preprocessing_reproduces_saved_processed_artifacts():
    loader = HeartDataLoader(RAW_PATH)
    cleaned = loader.clean()
    features, _ = loader.split_features_target(cleaned)
    saved = pd.read_csv(PROCESSED_DIR/"heart_dedup.csv")
    pd.testing.assert_frame_equal(cleaned, saved, check_dtype=False)
    X_now = pd.read_csv(PROCESSED_DIR/"X_manhattan.csv")
    rebuilt, names, _ = FeaturePreprocessor().fit_transform(features)
    assert names == X_now.columns.tolist()
    np.testing.assert_allclose(rebuilt, X_now.to_numpy(dtype=float), rtol=0, atol=1e-12)
    G_now = np.load(PROCESSED_DIR/"gower_distance.npy", mmap_mode="r")
    np.testing.assert_allclose(DistanceCalculator.gower(features), G_now, rtol=0, atol=1e-12)
    return "Rebuilt Manhattan and Gower match the saved inputs"


def test_read_only_full_data_custom_smoke():
    X = pd.read_csv(PROCESSED_DIR/"X_manhattan.csv").to_numpy(dtype=float)
    ref = pd.read_csv(PROCESSED_DIR/"reference.csv")
    saved = pd.read_csv(METRICS_DIR/"labels.csv")["manhattan_k2"].to_numpy()
    model = KMedoids(2, metric="manhattan", random_state=42).fit(X)
    assert len(model.labels_) == len(saved) == 2283
    assert model.converged_ and model.medoid_indices_ == [2018,89]
    assert model.cluster_sizes_ == [2072,211]
    assert np.isclose(model.inertia_,15072.767679141063)
    assert np.isclose(ClusterEvaluator.silhouette(X, model.labels_),0.2125827422943953)
    for column in ref.columns:
        scores = ClusterEvaluator.posthoc_check(model.labels_, ref[column])
        assert 0 <= scores["purity"] <= 1 and -1 <= scores["ARI"] <= 1
    agreement = ClusterEvaluator.compare_labelings(model.labels_, saved)
    assert -1 <= agreement <= 1
    return f"medoids={model.medoid_indices_}; objective={model.inertia_:.6f}; custom/library ARI={agreement:.6f}"


def test_30_records_match_notebook_02():
    matrix_path = PROCESSED_DIR/"X_manhattan.csv"
    X = pd.read_csv(matrix_path).iloc[:30].to_numpy(dtype=float)
    summary = json.loads((EXAMPLE_DIR/"summary.json").read_text(encoding="utf-8"))
    assignments = pd.read_csv(EXAMPLE_DIR/"assignments.csv")
    assert X.shape == (30,21) and summary["n_transformed_features"] == 21
    assert summary["matrix_sha256"] == sha256(matrix_path.read_bytes()).hexdigest()
    assert summary["held_out_columns"] == [*REFERENCE_COLS,TARGET_COL]
    custom = KMedoids(2, metric="manhattan", initial_medoids=[0,15]).fit(X)
    assert custom.history_[0]["cluster_sizes"] == (10,20)
    np.testing.assert_allclose(
        [custom.history_[0]["objective"], custom.inertia_],
        [226.8629313016385,175.25744477502278], rtol=0, atol=1e-9,
    )
    assert custom.medoid_indices_ == [24,17]
    assert set(custom.medoid_indices_) == set(summary["medoid_indices_zero_based"])
    assert ClusterEvaluator.compare_labelings(custom.labels_, assignments["cluster"].to_numpy()) == 1
    assert assignments["cluster"].tolist() == summary["labels"]
    np.testing.assert_allclose([custom.inertia_, assignments["assigned_distance"].sum()], summary["inertia"])
    return f"R25/R18, {custom.inertia_:.6f}; ARI=1 against Notebook 02 assignments"


for name, fn in [
    ("saved artifact/config alignment",test_saved_artifacts_and_config_are_aligned),
    ("rebuild preprocessing vs saved artifacts",test_preprocessing_reproduces_saved_processed_artifacts),
    ("custom full-data k=2 smoke",test_read_only_full_data_custom_smoke),
    ("30 records vs Notebook 02",test_30_records_match_notebook_02),
]:
    test_case("ARTIFACTS/INTEGRATION / " + name, fn)


def test_no_protected_artifact_was_modified():
    changed = [str(p.relative_to(PROJECT_ROOT)) for p,digest in FILE_HASHES_BEFORE.items()
               if not p.exists() or sha256(p.read_bytes()).hexdigest() != digest]
    assert not changed, f"Input artifacts changed: {changed}"
    return f"{len(FILE_HASHES_BEFORE)} files preserved by SHA-256"


test_case("READ-ONLY / protected artifacts",test_no_protected_artifact_was_modified)

PASS | ARTIFACTS/INTEGRATION / saved artifact/config alignment | 2,283 aligned rows, 21 encoded columns, cp/target separate
PASS | ARTIFACTS/INTEGRATION / rebuild preprocessing vs saved artifacts | Rebuilt Manhattan and Gower match the saved inputs
PASS | ARTIFACTS/INTEGRATION / custom full-data k=2 smoke | medoids=[2018, 89]; objective=15072.767679; custom/library ARI=-0.018271
PASS | ARTIFACTS/INTEGRATION / 30 records vs Notebook 02 | R25/R18, 175.257445; ARI=1 against Notebook 02 assignments
PASS | READ-ONLY / protected artifacts | 16 files preserved by SHA-256


True

## 6. Tổng hợp kết quả

Các dòng WARN được giữ lại để theo dõi quy tắc mã và chế độ lọc nguồn. FAIL cho biết phép kiểm tra bắt buộc cần sửa.

In [7]:
results_df = pd.DataFrame(TEST_RESULTS)
display(results_df)
counts = results_df["status"].value_counts()
print(f"PASS={counts.get('PASS', 0)} | WARN={counts.get('WARN', 0)} | FAIL={counts.get('FAIL', 0)}")
if counts.get("FAIL", 0):
    raise AssertionError("Có phép kiểm tra bắt buộc thất bại.")

,test,status,required,detail,note
0,CUSTOM / 30-record hand calculation,PASS,True,"medoids=[7, 22]; objective=112.0; iterations=2",
1,CUSTOM / medoid must be observed record,PASS,True,"medoid rows=[0, 3] are members of X",
2,CUSTOM / random seed reproducibility,PASS,True,"same seed -> medoids=[6, 2, 13], labels/object...",
3,CUSTOM / Euclidean and Manhattan distances,PASS,True,Euclidean and Manhattan produce expected label...,
4,CUSTOM / fit_predict and predict,PASS,True,fit_predict matches labels_; predict(X_train/n...,
5,CUSTOM / predict input errors,PASS,True,predict before fit and wrong feature count rej...,
6,CUSTOM / constructor validation,PASS,True,"invalid k type/value, max_iter, and metric rej...",
7,CUSTOM / fit data/k/initial medoid validation,PASS,True,"empty/ragged/non-numeric/non-finite data, bad ...",
8,CUSTOM / max_iter and final recomputation,PASS,True,"max_iter=1 -> n_iter=1, converged=False; final...",
9,CUSTOM / duplicate rows and assignment tie,PASS,True,duplicate points preserved; assignment tie res...,


PASS=29 | WARN=2 | FAIL=0
